## 1. 설정

In [ ]:
import os, glob
import numpy as np
import pandas as pd

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/MLB_pitcher'
    STATCAST_GLOB = f'{BASE}/statcast/statcast_*.parquet'  # 경로 맞게 조정
    OUT = f'{BASE}/data/game_targets.parquet'
else:
    BASE = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측\0_data'
    STATCAST_GLOB = os.path.join(BASE, 'statcast', 'statcast_*.parquet')
    OUT = os.path.join(BASE, 'data', 'game_targets.parquet')

X_PITCHES = 15      # X구간 = 경기 앞 15구 (그 이후가 Y구간)
MIN_SWINGS = 20     # whiff 분모 최소
MIN_Y_BF   = 5      # Y구간 최소 타자수

files = sorted(glob.glob(STATCAST_GLOB))
print('statcast 파일:', len(files), files[:2])

## 2. statcast 로드 + 선발 X/Y 구간 분리

X구간 = 경기 앞 15구. Y구간 = 그 이후. 경기별 타겟은 Y구간으로 계산.

In [ ]:
USE_COLS = ['game_pk','pitcher','game_year','inning','at_bat_number','pitch_number',
            'events','description','estimated_woba_using_speedangle',
            'post_bat_score','bat_score','inning_topbot']

dfs = []
for f in files:
    d = pd.read_parquet(f)
    keep = [c for c in USE_COLS if c in d.columns]
    d = d[keep].copy()
    if 'game_year' in d.columns:
        d = d.rename(columns={'game_year':'season'})
    dfs.append(d)
sc = pd.concat(dfs, ignore_index=True)
sc = sc.dropna(subset=['game_pk','pitcher','at_bat_number','pitch_number'])
sc[['game_pk','pitcher','at_bat_number','pitch_number']] = sc[['game_pk','pitcher','at_bat_number','pitch_number']].astype('int64')
print('투구:', len(sc), '경기수:', sc.groupby(['game_pk','pitcher']).ngroups)

# 경기 내 투구 순서 매기기 → 앞 X_PITCHES = X구간
sc = sc.sort_values(['game_pk','pitcher','at_bat_number','pitch_number'])
sc['pitch_rank'] = sc.groupby(['game_pk','pitcher']).cumcount() + 1
sc['zone'] = np.where(sc['pitch_rank'] <= X_PITCHES, 'X', 'Y')
y_zone = sc[sc['zone']=='Y'].copy()
print('Y구간 투구:', len(y_zone))

## 3. 경기별 타겟 4종 (whiff%, xwOBA, FIP, RA9) — Y구간 기준

In [ ]:
KEYS = ['game_pk','pitcher','season']
g = y_zone.groupby(KEYS)

# --- whiff% ---
desc = y_zone['description']
y_zone['_whiff'] = desc.isin(['swinging_strike','swinging_strike_blocked']).astype(int)
y_zone['_swing'] = desc.isin(['swinging_strike','swinging_strike_blocked','foul','foul_tip','hit_into_play']).astype(int)

# --- FIP 원천 (events) ---
ev = y_zone['events'].fillna('')
y_zone['_hr']  = (ev=='home_run').astype(int)
y_zone['_bb']  = ev.isin(['walk','intent_walk']).astype(int)
y_zone['_hbp'] = (ev=='hit_by_pitch').astype(int)
y_zone['_k']   = ev.isin(['strikeout','strikeout_double_play']).astype(int)
# 타자수(BF) = events 있는 행(타석 종료)
y_zone['_bf']  = (ev!='').astype(int)
# 실점: 타석 종료 시 bat_score 증가분 (근사 — RA)
if 'post_bat_score' in y_zone.columns and 'bat_score' in y_zone.columns:
    y_zone['_runs'] = (y_zone['post_bat_score'] - y_zone['bat_score']).clip(lower=0).fillna(0)
else:
    y_zone['_runs'] = 0

agg = g.agg(
    whiffs=('_whiff','sum'), swings=('_swing','sum'),
    xwoba_mean=('estimated_woba_using_speedangle','mean'),
    hr=('_hr','sum'), bb=('_bb','sum'), hbp=('_hbp','sum'), k=('_k','sum'),
    bf=('_bf','sum'), runs=('_runs','sum'),
    y_innings=('inning', lambda x: x.nunique()),  # Y구간이 걸친 이닝 수(근사 IP)
).reset_index()

# whiff%
agg = agg[agg['swings'] >= MIN_SWINGS].copy()
agg['y_whiff'] = (agg['whiffs'] / agg['swings']).round(4)
# xwOBA (Y구간 평균 기대 wOBA)
agg['y_xwoba'] = agg['xwoba_mean'].round(4)
# FIP = (13HR + 3(BB+HBP) - 2K)/IP + C  (C는 리그상수, 여기선 3.1 고정 근사)
FIP_CONST = 3.1
ip = agg['y_innings'].replace(0, np.nan)
agg['y_fip'] = ((13*agg['hr'] + 3*(agg['bb']+agg['hbp']) - 2*agg['k'])/ip + FIP_CONST).round(3)
# RA9 = 실점/이닝 * 9 (ERA 대용, 자책 구분 없는 실점)
agg['y_ra9'] = (agg['runs']/ip*9).round(3)
# BF 너무 적은 경기 제외
agg = agg[agg['bf'] >= MIN_Y_BF].copy()

game_y = agg[KEYS + ['y_whiff','y_xwoba','y_fip','y_ra9']].copy()
print('경기별 타겟:', game_y.shape)
print(game_y[['y_whiff','y_xwoba','y_fip','y_ra9']].describe().round(3).T.to_string())

## 5. 경기별 + 시즌별 합쳐 저장

In [ ]:
# game_y 를 최종 타겟으로 저장 (전부 경기단위 4종)
targets = game_y.rename(columns={'y_ra9': 'y_era'}).copy()  # RA9를 ERA로 명명(경기 실점/이닝×9)
targets.to_parquet(OUT, index=False)
print('저장:', OUT)
print('shape:', targets.shape)
print('
타겟 4종:', [c for c in targets.columns if c.startswith('y_')])
print('
타겟 분포:')
print(targets[['y_whiff','y_xwoba','y_fip','y_era']].describe().round(3).T.to_string())
targets.head(3)
